In [1]:
# ============================================================
# CELL 1: Mount Google Drive and Configure Project Paths
# ============================================================

from google.colab import drive
from pathlib import Path
import shutil
import sys
import torch

# ------------------------------------------------------------
# 1. Mount Google Drive
# ------------------------------------------------------------
drive.mount('/content/drive')

# ------------------------------------------------------------
# 2. Define canonical project paths
# ------------------------------------------------------------
DRIVE_ROOT = Path('/content/drive/MyDrive/Vi-SAFE-3B42')

LOCAL_ROOT = Path('/content/Vi-SAFE-3B42-Implementation')
LOCAL_DATA = Path('/content/RWF-2000')

SOURCE_APP = DRIVE_ROOT / 'app'
YOLO_WEIGHTS = DRIVE_ROOT / 'yolov8n.pt'

# Drive-backed artifacts
DRIVE_OUTPUT = DRIVE_ROOT / 'outputs'
DRIVE_CHECKPOINTS = DRIVE_OUTPUT / 'checkpoints'
DRIVE_LOGS = DRIVE_OUTPUT / 'logs'
DRIVE_METRICS = DRIVE_OUTPUT / 'metrics'

DRIVE_BACKUPS = DRIVE_ROOT / 'backups'
SOURCE_BACKUP = DRIVE_BACKUPS / 'v2_source_snapshots'

# ------------------------------------------------------------
# 3. Create required directories
# ------------------------------------------------------------
for directory in [
    LOCAL_ROOT,
    DRIVE_OUTPUT,
    DRIVE_CHECKPOINTS,
    DRIVE_LOGS,
    DRIVE_METRICS,
    DRIVE_BACKUPS,
    SOURCE_BACKUP
]:
    directory.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 4. Verify existing project files
# ------------------------------------------------------------
print('=' * 65)
print('VI-SAFE PROJECT CONFIGURATION')
print('=' * 65)

assert DRIVE_ROOT.is_dir(), f'Google Drive project folder not found: {DRIVE_ROOT}'

required_files = {
    'YOLOv8n weights': YOLO_WEIGHTS,
    'Original notebook': DRIVE_ROOT / 'RWF-2000-v1.ipynb',
    'New notebook': DRIVE_ROOT / 'RWF-2000-v2.ipynb',
    'Requirements': DRIVE_ROOT / 'requirements.txt',
}

for name, path in required_files.items():
    print(f'{name:22}: {"FOUND" if path.is_file() else "MISSING"}')
    if name == 'YOLOv8n weights' and not path.is_file():
        raise FileNotFoundError(path)

print('\\nOriginal source directories:')
for name in ['app', 'app_simple', 'mmaction2']:
    path = DRIVE_ROOT / name
    print(f'{name:22}: {"FOUND" if path.is_dir() else "NOT FOUND"}')

print('\\n' + '=' * 65)
print('RUNTIME INFORMATION')
print('=' * 65)

print('Python version       :', sys.version.split()[0])
print('PyTorch version      :', torch.__version__)
print('CUDA available       :', torch.cuda.is_available())

if torch.cuda.is_available():
    print('GPU                  :', torch.cuda.get_device_name(0))

print('Free /content space  :',
      round(shutil.disk_usage('/content').free / (1024**3), 2), 'GB')

# ------------------------------------------------------------
# 5. Dataset status (must remain on local Colab storage)
# ------------------------------------------------------------
print('\\n' + '=' * 65)
print('DATASET STATUS')
print('=' * 65)

if LOCAL_DATA.is_dir():
    print('Dataset already exists:', LOCAL_DATA)
else:
    print('Dataset not staged yet.')
    print('Expected local location:', LOCAL_DATA)

print('\\n' + '=' * 65)
print('PATH CONFIGURATION COMPLETE')
print('=' * 65)

print('Drive root       :', DRIVE_ROOT)
print('Local source     :', LOCAL_ROOT)
print('Local dataset    :', LOCAL_DATA)
print('Checkpoints      :', DRIVE_CHECKPOINTS)
print('Logs             :', DRIVE_LOGS)
print('Metrics          :', DRIVE_METRICS)
print('Backups          :', DRIVE_BACKUPS)

Mounted at /content/drive
VI-SAFE PROJECT CONFIGURATION
YOLOv8n weights       : FOUND
Original notebook     : FOUND
New notebook          : FOUND
Requirements          : FOUND
\nOriginal source directories:
app                   : FOUND
app_simple            : FOUND
mmaction2             : FOUND
\n=================================================================
RUNTIME INFORMATION
Python version       : 3.13.16
PyTorch version      : 2.14.1+cu130
CUDA available       : True
GPU                  : Tesla T4
Free /content space  : 112.54 GB
\n=================================================================
DATASET STATUS
Dataset not staged yet.
Expected local location: /content/RWF-2000
\n=================================================================
PATH CONFIGURATION COMPLETE
Drive root       : /content/drive/MyDrive/Vi-SAFE-3B42
Local source     : /content/Vi-SAFE-3B42-Implementation
Local dataset    : /content/RWF-2000
Checkpoints      : /content/drive/MyDrive/Vi-SAFE-3B42/output

In [2]:
# ============================================================
# CELL 2: AUDIT ORIGINAL VI-SAFE SOURCE CODE
# ============================================================

from pathlib import Path

APP_DIR = DRIVE_ROOT / "app"
APP_SIMPLE_DIR = DRIVE_ROOT / "app_simple"
MMACTION_DIR = DRIVE_ROOT / "mmaction2"
REQUIREMENTS_FILE = DRIVE_ROOT / "requirements.txt"

def format_size(size_bytes):
    if size_bytes >= 1024**3:
        return f"{size_bytes / 1024**3:.2f} GB"
    if size_bytes >= 1024**2:
        return f"{size_bytes / 1024**2:.2f} MB"
    if size_bytes >= 1024:
        return f"{size_bytes / 1024:.2f} KB"
    return f"{size_bytes} bytes"

def inspect_directory(directory, max_files=100):
    if not directory.exists():
        print(f"\n{directory.name}: NOT FOUND")
        return

    files = sorted(p for p in directory.rglob("*") if p.is_file())
    total_size = sum(p.stat().st_size for p in files)

    print(f"\n{'=' * 65}")
    print(f"DIRECTORY: {directory}")
    print(f"Files      : {len(files)}")
    print(f"Total size : {format_size(total_size)}")
    print("=" * 65)

    for file_path in files[:max_files]:
        relative_path = file_path.relative_to(directory)
        print(f"{relative_path}  [{format_size(file_path.stat().st_size)}]")

    if len(files) > max_files:
        print(f"... and {len(files) - max_files} more files")

# Inspect the two implementation directories.
inspect_directory(APP_DIR)
inspect_directory(APP_SIMPLE_DIR)

# Inspect only the top-level structure of the MMAction2 directory.
print("\n" + "=" * 65)
print("MMACTION2 DIRECTORY OVERVIEW")
print("=" * 65)

if MMACTION_DIR.exists():
    for item in sorted(MMACTION_DIR.iterdir()):
        kind = "DIR " if item.is_dir() else "FILE"
        print(f"{kind}  {item.name}")
else:
    print("MMAction2 directory not found.")

# Check important source files.
important_files = [
    APP_DIR / "encoder_yolo.py",
    APP_DIR / "yolo_tsn.py",
    APP_DIR / "dataset_rwf2000.py",
    APP_DIR / "main.py",
]

print("\n" + "=" * 65)
print("IMPORTANT SOURCE FILE CHECK")
print("=" * 65)

for file_path in important_files:
    status = "FOUND" if file_path.is_file() else "MISSING"
    print(f"{file_path.relative_to(DRIVE_ROOT)} : {status}")

# Display requirements without installing anything.
print("\n" + "=" * 65)
print("REQUIREMENTS.TXT")
print("=" * 65)

if REQUIREMENTS_FILE.exists():
    print(REQUIREMENTS_FILE.read_text(encoding="utf-8"))
else:
    print("requirements.txt not found.")

# Confirm the YOLO weights file.
print("\nYOLOv8n weights:")
print("Path:", YOLO_WEIGHTS)
print("Size:", format_size(YOLO_WEIGHTS.stat().st_size)
      if YOLO_WEIGHTS.exists() else "NOT FOUND")

print("\nSOURCE AUDIT COMPLETE")


DIRECTORY: /content/drive/MyDrive/Vi-SAFE-3B42/app
Files      : 5
Total size : 49.26 KB
__pycache__/encoder_yolo.cpython-313.pyc  [12.06 KB]
dataset_rwf2000.py  [5.22 KB]
encoder_yolo.py  [11.86 KB]
main.py  [14.81 KB]
yolo_tsn.py  [5.31 KB]

DIRECTORY: /content/drive/MyDrive/Vi-SAFE-3B42/app_simple
Files      : 4
Total size : 24.92 KB
dataset_rwf2000.py  [6.39 KB]
encoder_yolo.py  [4.39 KB]
main.py  [9.99 KB]
yolo_tsn.py  [4.15 KB]

MMACTION2 DIRECTORY OVERVIEW
DIR   .circleci
DIR   .github
FILE  .gitignore
FILE  .owners.yml
FILE  .pre-commit-config.yaml
FILE  .pylintrc
FILE  .readthedocs.yml
FILE  CITATION.cff
FILE  LICENSE
FILE  MANIFEST.in
FILE  README.md
FILE  README_zh-CN.md
DIR   configs
FILE  dataset-index.yml
DIR   demo
DIR   docker
DIR   docs
DIR   mmaction
FILE  model-index.yml
DIR   projects
DIR   requirements
FILE  requirements.txt
DIR   resources
FILE  setup.cfg
FILE  setup.py
DIR   tests
DIR   tools
FILE  tsn_imagenet-pretrained-r50_8xb32-1x1x8-100e_kinetics400-rgb.py



In [3]:
# ============================================================
# CELL 3: INSPECT MODEL, DATASET AND FRAMEWORK INTERFACES
# ============================================================

import ast
from pathlib import Path

SOURCE_FILES = [
    APP_DIR / "encoder_yolo.py",
    APP_DIR / "yolo_tsn.py",
    APP_DIR / "dataset_rwf2000.py",
    APP_DIR / "main.py",
]

KEYWORDS = (
    "fight", "nonfight", "label", "num_segments",
    "temporal", "TSN", "YOLO", "mmaction",
    "checkpoint", "freeze", "batch_size",
)

for file_path in SOURCE_FILES:
    print("\n" + "=" * 75)
    print(f"FILE: {file_path.name}")
    print("=" * 75)

    source = file_path.read_text(encoding="utf-8")
    lines = source.splitlines()
    tree = ast.parse(source)

    print("\n[IMPORTS]")
    for node in ast.walk(tree):
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            print(ast.unparse(node))

    print("\n[CLASSES AND FUNCTIONS]")
    for node in ast.walk(tree):
        if isinstance(node, (ast.ClassDef, ast.FunctionDef, ast.AsyncFunctionDef)):
            if isinstance(node, ast.ClassDef):
                print(f"class {node.name}")
            else:
                args = ast.unparse(node.args)
                print(f"def {node.name}({args})")

    print("\n[IMPORTANT CONFIGURATION / LABEL LINES]")
    matches = []
    for line_number, line in enumerate(lines, start=1):
        if any(keyword.lower() in line.lower() for keyword in KEYWORDS):
            matches.append((line_number, line.strip()))

    for line_number, line in matches[:60]:
        print(f"{line_number:4}: {line}")

    if len(matches) > 60:
        print(f"... {len(matches) - 60} additional matching lines omitted")

# ------------------------------------------------------------
# Inspect MMAction2 package version and required directories.
# ------------------------------------------------------------
print("\n" + "=" * 75)
print("MMACTION2 FRAMEWORK CHECK")
print("=" * 75)

version_file = MMACTION_DIR / "mmaction" / "version.py"
init_file = MMACTION_DIR / "mmaction" / "__init__.py"

for path in [version_file, init_file]:
    print(f"{path.relative_to(DRIVE_ROOT)}:",
          "FOUND" if path.exists() else "MISSING")

if version_file.exists():
    print("\nVersion information:")
    print(version_file.read_text(encoding="utf-8")[:2000])

for relative_path in [
    "mmaction/datasets",
    "mmaction/models",
    "mmaction/apis",
    "configs",
]:
    path = MMACTION_DIR / relative_path
    print(f"{relative_path:25}:",
          "FOUND" if path.exists() else "MISSING")

print("\nINTERFACE AUDIT COMPLETE")


FILE: encoder_yolo.py

[IMPORTS]
import torch
from torch import nn
import torch.nn.functional as F
from ultralytics import YOLO

[CLASSES AND FUNCTIONS]
class BboxTransformerAggregator
class YOLOEncoder
def __init__(self, input_dim=13, hidden_dim=16, aggregator_out_dim=128, num_heads=2, use_cls_token=False)
def _init_weights(self)
def forward(self, bbox_embeddings)
def __init__(self, yolo_weights_path='yolov8n.pt', device='cuda', freeze_yolo=True, conf_thresh=0.3, top_k=5, class_embed_dim=8, use_attention=True, use_frame_residual=True)
def train(self, mode: bool=True)
def forward(self, frames)

[IMPORTANT CONFIGURATION / LABEL LINES]
   4: from ultralytics import YOLO
  98: class YOLOEncoder(nn.Module):
 102: yolo_weights_path='yolov8n.pt',
 104: freeze_yolo=True,
 112: :param yolo_weights_path: YOLOv8 weights
 114: :param freeze_yolo: if True, YOLO backbone is frozen
 121: and sum it with YOLO aggregator output (i.e. "residual").
 129: self.freeze_yolo = freeze_yolo
 131: # Load YOLO

In [4]:
# ============================================================
# CELL 4: BACKUP ORIGINAL CODE AND STAGE LOCAL WORKSPACE
# ============================================================

import shutil
from datetime import datetime
from pathlib import Path

# ------------------------------------------------------------
# 1. Create a timestamped backup on Google Drive.
# ------------------------------------------------------------
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

SNAPSHOT_DIR = DRIVE_BACKUPS / "v2_source_snapshots" / f"pre_v2_{timestamp}"
SNAPSHOT_DIR.mkdir(parents=True, exist_ok=False)

def copy_source(source, destination):
    if not source.exists():
        raise FileNotFoundError(f"Required source not found: {source}")

    shutil.copytree(
        source,
        destination,
        ignore=shutil.ignore_patterns("__pycache__", "*.pyc")
    )

copy_source(APP_DIR, SNAPSHOT_DIR / "app")
copy_source(APP_SIMPLE_DIR, SNAPSHOT_DIR / "app_simple")

if REQUIREMENTS_FILE.exists():
    shutil.copy2(REQUIREMENTS_FILE, SNAPSHOT_DIR / "requirements.txt")

print("Original source backup created:")
print(SNAPSHOT_DIR)

# ------------------------------------------------------------
# 2. Prepare the local workspace in Colab.
# ------------------------------------------------------------
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

LOCAL_APP = LOCAL_ROOT / "app"
LOCAL_APP_SIMPLE = LOCAL_ROOT / "app_simple"

# These are working copies only. The originals remain in Drive.
for target in [LOCAL_APP, LOCAL_APP_SIMPLE]:
    if target.exists():
        shutil.rmtree(target)

copy_source(APP_DIR, LOCAL_APP)
copy_source(APP_SIMPLE_DIR, LOCAL_APP_SIMPLE)

# Add package initializers to the local working copies.
# This does not modify the original source.
(LOCAL_APP / "__init__.py").touch(exist_ok=True)
(LOCAL_APP_SIMPLE / "__init__.py").touch(exist_ok=True)

# Copy requirements for later dependency review.
if REQUIREMENTS_FILE.exists():
    shutil.copy2(REQUIREMENTS_FILE, LOCAL_ROOT / "requirements.txt")

# ------------------------------------------------------------
# 3. Copy YOLOv8n weights to local storage.
# ------------------------------------------------------------
LOCAL_YOLO_WEIGHTS = LOCAL_ROOT / "yolov8n.pt"

if not YOLO_WEIGHTS.exists():
    raise FileNotFoundError(f"YOLO weights not found: {YOLO_WEIGHTS}")

shutil.copy2(YOLO_WEIGHTS, LOCAL_YOLO_WEIGHTS)

# ------------------------------------------------------------
# 4. Verify staged files.
# ------------------------------------------------------------
print("\n" + "=" * 65)
print("LOCAL WORKSPACE READY")
print("=" * 65)

for directory in [LOCAL_APP, LOCAL_APP_SIMPLE]:
    files = sorted(p for p in directory.rglob("*") if p.is_file())
    print(f"\n{directory}")
    print(f"Files staged: {len(files)}")
    for file_path in files:
        print(" -", file_path.relative_to(LOCAL_ROOT))

print("\nYOLO weights:", LOCAL_YOLO_WEIGHTS)
print("YOLO weights exist:", LOCAL_YOLO_WEIGHTS.exists())

print("\nOriginal backup exists:", SNAPSHOT_DIR.exists())
print("Dataset copied:", False)
print("Dataset location reserved:", LOCAL_DATA)

print("\nCELL 4 COMPLETE")

Original source backup created:
/content/drive/MyDrive/Vi-SAFE-3B42/backups/v2_source_snapshots/pre_v2_20261111_164830

LOCAL WORKSPACE READY

/content/Vi-SAFE-3B42-Implementation/app
Files staged: 5
 - app/__init__.py
 - app/dataset_rwf2000.py
 - app/encoder_yolo.py
 - app/main.py
 - app/yolo_tsn.py

/content/Vi-SAFE-3B42-Implementation/app_simple
Files staged: 5
 - app_simple/__init__.py
 - app_simple/dataset_rwf2000.py
 - app_simple/encoder_yolo.py
 - app_simple/main.py
 - app_simple/yolo_tsn.py

YOLO weights: /content/Vi-SAFE-3B42-Implementation/yolov8n.pt
YOLO weights exist: True

Original backup exists: True
Dataset copied: False
Dataset location reserved: /content/RWF-2000

CELL 4 COMPLETE


In [5]:
# ============================================================
# CELL 5: CHECK INSTALLED DEPENDENCIES AND COMPATIBILITY
# ============================================================

import sys
import importlib.metadata as metadata
import importlib.util
import subprocess
import torch

print("=" * 70)
print("PYTHON / PYTORCH / CUDA")
print("=" * 70)

print("Python       :", sys.version.split()[0])
print("PyTorch      :", torch.__version__)
print("CUDA version :", torch.version.cuda)
print("CUDA usable  :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU          :", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# Installed package versions
# ------------------------------------------------------------
packages = [
    "torch",
    "torchvision",
    "torchaudio",
    "ultralytics",
    "mmengine",
    "mmcv",
    "mmcv-lite",
    "mmaction2",
    "numpy",
    "opencv-python",
    "decord",
    "einops",
    "tqdm",
    "wandb",
]

print("\n" + "=" * 70)
print("INSTALLED PACKAGE VERSIONS")
print("=" * 70)

for package in packages:
    try:
        version = metadata.version(package)
        print(f"{package:18}: {version}")
    except metadata.PackageNotFoundError:
        print(f"{package:18}: NOT INSTALLED")

# ------------------------------------------------------------
# Check Python module availability
# ------------------------------------------------------------
modules = [
    "torch",
    "torchvision",
    "ultralytics",
    "mmengine",
    "mmcv",
    "mmaction",
    "cv2",
    "decord",
    "einops",
]

print("\n" + "=" * 70)
print("MODULE AVAILABILITY")
print("=" * 70)

for module in modules:
    try:
        spec = importlib.util.find_spec(module)
        print(f"{module:18}:",
              spec.origin if spec else "NOT FOUND")
    except Exception as exc:
        print(f"{module:18}: CHECK ERROR - {exc}")

# ------------------------------------------------------------
# Check torchvision import and CUDA operator compatibility.
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("TORCHVISION IMPORT TEST")
print("=" * 70)

try:
    import torchvision
    print("torchvision import : SUCCESS")
    print("torchvision version:", torchvision.__version__)

    try:
        from torchvision.ops import nms
        boxes = torch.tensor(
            [[0., 0., 10., 10.], [1., 1., 9., 9.]],
            device="cpu"
        )
        scores = torch.tensor([0.9, 0.8])
        result = nms(boxes, scores, 0.5)
        print("NMS operator       : SUCCESS")
    except Exception as exc:
        print("NMS operator       : FAILED")
        print("Reason             :", repr(exc))

except Exception as exc:
    print("torchvision import : FAILED")
    print("Reason             :", repr(exc))

# ------------------------------------------------------------
# Check dependency consistency without installing anything.
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("PIP DEPENDENCY CHECK")
print("=" * 70)

result = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    capture_output=True,
    text=True
)

print(result.stdout.strip() or "No dependency conflicts reported.")

if result.stderr.strip():
    print("Warnings:")
    print(result.stderr.strip())

print("\nCELL 5 COMPLETE")

PYTHON / PYTORCH / CUDA
Python       : 3.13.16
PyTorch      : 2.14.1+cu130
CUDA version : 13.0
CUDA usable  : True
GPU          : Tesla T4

INSTALLED PACKAGE VERSIONS
torch             : 2.14.1+cu130
torchvision       : 0.29.1+cu130
torchaudio        : 2.11.0+cu130
ultralytics       : NOT INSTALLED
mmengine          : NOT INSTALLED
mmcv              : NOT INSTALLED
mmcv-lite         : NOT INSTALLED
mmaction2         : NOT INSTALLED
numpy             : 2.4.6
opencv-python     : 5.0.0.93
decord            : NOT INSTALLED
einops            : 0.8.2
tqdm              : 4.67.3
wandb             : 0.28.1

MODULE AVAILABILITY
torch             : /usr/local/lib/python3.13/dist-packages/torch/__init__.py
torchvision       : /usr/local/lib/python3.13/dist-packages/torchvision/__init__.py
ultralytics       : NOT FOUND
mmengine          : NOT FOUND
mmcv              : NOT FOUND
mmaction          : NOT FOUND
cv2               : None
decord            : NOT FOUND
einops            : /usr/local/lib/py

In [6]:
# ============================================================
# CELL 6: GET COMPLETE MMACTION2 V1.2.0 SOURCE
# ============================================================

import subprocess
from pathlib import Path

LOCAL_MMACTION = Path("/content/mmaction2_full_v1_2_0")
MMACTION_REPO = "https://github.com/open-mmlab/mmaction2.git"

# ------------------------------------------------------------
# 1. Clone the exact version if it is not already available.
# ------------------------------------------------------------
if not LOCAL_MMACTION.exists():
    print("Cloning MMAction2 v1.2.0 into local Colab storage...")

    subprocess.run(
        [
            "git", "clone",
            "--depth", "1",
            "--branch", "v1.2.0",
            MMACTION_REPO,
            str(LOCAL_MMACTION),
        ],
        check=True,
    )
else:
    print("Local MMAction2 directory already exists; reusing it.")

# ------------------------------------------------------------
# 2. Verify the version and important package directories.
# ------------------------------------------------------------
version_file = LOCAL_MMACTION / "mmaction" / "version.py"

if not version_file.exists():
    raise FileNotFoundError(
        f"MMAction2 version file missing: {version_file}"
    )

version_text = version_file.read_text(encoding="utf-8")
print("\nVersion file:")
print(version_text)

required_paths = [
    "mmaction/__init__.py",
    "mmaction/registry.py",
    "mmaction/datasets",
    "mmaction/models",
    "mmaction/utils",
    "configs/recognition/tsn",
]

print("\n" + "=" * 65)
print("MMAction2 SOURCE VALIDATION")
print("=" * 65)

all_found = True

for relative_path in required_paths:
    exists = (LOCAL_MMACTION / relative_path).exists()
    print(f"{relative_path:35}: {'FOUND' if exists else 'MISSING'}")
    all_found = all_found and exists

# ------------------------------------------------------------
# 3. Record the exact Git revision.
# ------------------------------------------------------------
revision = subprocess.run(
    ["git", "-C", str(LOCAL_MMACTION), "rev-parse", "--short", "HEAD"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()

print("\nGit revision:", revision)
print("All required source paths found:", all_found)

if not all_found:
    raise RuntimeError(
        "The MMAction2 source tree is incomplete. Do not install it yet."
    )

print("\nMMAction2 source is ready for dependency compatibility testing.")
print("No packages were installed or modified.")
print("\nCELL 6 COMPLETE")

Cloning MMAction2 v1.2.0 into local Colab storage...

Version file:
# Copyright (c) Open-MMLab. All rights reserved.

__version__ = '1.2.0'


def parse_version_info(version_str: str):
    """Parse a version string into a tuple.

    Args:
        version_str (str): The version string.
    Returns:
        tuple[int or str]: The version info, e.g., "1.3.0" is parsed into
            (1, 3, 0), and "2.0.0rc1" is parsed into (2, 0, 0, 'rc1').
    """
    version_info = []
    for x in version_str.split('.'):
        if x.isdigit():
            version_info.append(int(x))
        elif x.find('rc') != -1:
            patch_version = x.split('rc')
            version_info.append(int(patch_version[0]))
            version_info.append(f'rc{patch_version[1]}')
    return tuple(version_info)


version_info = parse_version_info(__version__)


MMAction2 SOURCE VALIDATION
mmaction/__init__.py               : FOUND
mmaction/registry.py               : FOUND
mmaction/datasets                  : FOUND

In [7]:
# ============================================================
# CELL 7: INSPECT MMACTION2 REQUIREMENTS AND TSN CONFIGURATION
# ============================================================

from pathlib import Path
import re

# ------------------------------------------------------------
# 1. Read MMAction2 dependency requirement files.
# ------------------------------------------------------------
print("=" * 75)
print("MMACTION2 DEPENDENCY REQUIREMENTS")
print("=" * 75)

requirement_files = [
    "requirements/build.txt",
    "requirements/runtime.txt",
    "requirements/mminstall.txt",
    "requirements/optional.txt",
]

for relative_path in requirement_files:
    path = LOCAL_MMACTION / relative_path

    print(f"\n--- {relative_path} ---")

    if path.exists():
        print(path.read_text(encoding="utf-8"))
    else:
        print("File not found.")

# ------------------------------------------------------------
# 2. Inspect MMAction2 version compatibility checks.
# ------------------------------------------------------------
print("\n" + "=" * 75)
print("MMACTION2 VERSION COMPATIBILITY CHECK")
print("=" * 75)

init_file = LOCAL_MMACTION / "mmaction" / "__init__.py"

if init_file.exists():
    lines = init_file.read_text(encoding="utf-8").splitlines()

    for number, line in enumerate(lines[:100], start=1):
        if any(term in line.lower() for term in
               ["mmcv", "mmengine", "version", "assert"]):
            print(f"{number:4}: {line}")

# ------------------------------------------------------------
# 3. Identify the TSN configuration expected by main.py.
# ------------------------------------------------------------
print("\n" + "=" * 75)
print("TSN CONFIGURATION REFERENCES IN APP/MAIN.PY")
print("=" * 75)

main_file = LOCAL_APP / "main.py"
main_lines = main_file.read_text(encoding="utf-8").splitlines()

for number, line in enumerate(main_lines, start=1):
    if "mmaction_cfg" in line.lower():
        start = max(0, number - 3)
        end = min(len(main_lines), number + 3)

        for index in range(start, end):
            print(f"{index + 1:4}: {main_lines[index]}")
        print("-" * 40)

# ------------------------------------------------------------
# 4. List available TSN configuration files.
# ------------------------------------------------------------
print("\n" + "=" * 75)
print("AVAILABLE TSN CONFIGURATIONS")
print("=" * 75)

tsn_config_dir = LOCAL_MMACTION / "configs" / "recognition" / "tsn"

if tsn_config_dir.exists():
    tsn_configs = sorted(tsn_config_dir.glob("*.py"))

    for config in tsn_configs:
        print(config.name)

    print("\nTotal TSN configs:", len(tsn_configs))
else:
    print("TSN config directory not found.")

# ------------------------------------------------------------
# 5. Inspect OpenCV import status.
# ------------------------------------------------------------
print("\n" + "=" * 75)
print("OPENCV IMPORT DIAGNOSTIC")
print("=" * 75)

try:
    import cv2
    print("OpenCV import successful.")
    print("OpenCV version:", cv2.__version__)
    print("OpenCV location:", cv2.__file__)
except Exception as exc:
    print("OpenCV import failed.")
    print("Error:", repr(exc))

print("\nCELL 7 COMPLETE")

MMACTION2 DEPENDENCY REQUIREMENTS

--- requirements/build.txt ---
decord >= 0.4.1
einops
matplotlib
numpy
opencv-contrib-python
Pillow
scipy
torch>=1.3


--- requirements/runtime.txt ---
File not found.

--- requirements/mminstall.txt ---
mmcv>=2.0.0rc4,<2.2.0
mmengine>=0.7.1,<1.0.0


--- requirements/optional.txt ---
av>=9.0
future
imgaug
librosa
lmdb
moviepy
openai-clip
packaging
pims
PyTurboJPEG
soundfile
tensorboard
wandb


MMACTION2 VERSION COMPATIBILITY CHECK
   2: import mmcv
   3: import mmengine
   4: from mmengine.utils import digit_version
   6: from .version import __version__
   8: mmcv_minimum_version = '2.0.0rc4'
   9: mmcv_maximum_version = '2.2.0'
  10: mmcv_version = digit_version(mmcv.__version__)
  12: mmengine_minimum_version = '0.7.1'
  13: mmengine_maximum_version = '1.0.0'
  14: mmengine_version = digit_version(mmengine.__version__)
  16: assert (digit_version(mmcv_minimum_version) <= mmcv_version
  17:         < digit_version(mmcv_maximum_version)), \
  18:    

In [8]:
# ============================================================
# CELL 8: DRY-RUN DEPENDENCIES + INSPECT TSN CONFIG
# ============================================================

import json
import subprocess
import sys
from pathlib import Path

# ------------------------------------------------------------
# 1. Resolve dependencies without installing them.
# ------------------------------------------------------------
packages_to_test = [
    "mmengine==0.10.7",
    "mmcv-lite==2.1.0",
    "ultralytics",
]

report_path = Path("/content/visafe_pip_dry_run.json")

command = [
    sys.executable, "-m", "pip", "install",
    "--dry-run",
    "--report", str(report_path),
    *packages_to_test,
]

print("=" * 70)
print("PIP DEPENDENCY DRY RUN")
print("=" * 70)
print("Command:", " ".join(command))

result = subprocess.run(
    command,
    capture_output=True,
    text=True,
)

print("\nPIP OUTPUT:")
print(result.stdout[-12000:] if result.stdout else "(No standard output)")

if result.stderr:
    print("\nPIP DIAGNOSTICS:")
    print(result.stderr[-6000:])

print("\nDry-run exit code:", result.returncode)

# ------------------------------------------------------------
# 2. Summarize the proposed installation changes.
# ------------------------------------------------------------
if result.returncode == 0 and report_path.exists():
    report = json.loads(report_path.read_text(encoding="utf-8"))

    print("\n" + "=" * 70)
    print("PROPOSED PACKAGE INSTALLATIONS")
    print("=" * 70)

    proposed = report.get("install", [])

    for item in proposed:
        metadata = item.get("metadata", {})
        name = metadata.get("name", "unknown")
        version = metadata.get("version", "unknown")
        print(f"{name:30} {version}")

    print("\nTotal packages proposed:", len(proposed))

    # Protect the existing GPU stack.
    protected = {"torch", "torchvision", "torchaudio"}
    planned_protected = []

    for item in proposed:
        name = item.get("metadata", {}).get("name", "").lower()
        if name in protected:
            planned_protected.append(
                (name, item.get("metadata", {}).get("version"))
            )

    if planned_protected:
        print("\nWARNING: Protected GPU packages appear in install plan:")
        for name, version in planned_protected:
            print(name, version)
    else:
        print("\nGPU package protection check: PASS")
else:
    print("\nDependency resolution did not complete successfully.")
    print("Do not install packages yet.")

# ------------------------------------------------------------
# 3. Inspect the intended 8-segment TSN configuration.
# ------------------------------------------------------------
CONFIG_FILE = (
    LOCAL_MMACTION
    / "configs/recognition/tsn/"
    / "tsn_imagenet-pretrained-r50_8xb32-1x1x8-100e_kinetics400-rgb.py"
)

print("\n" + "=" * 70)
print("TSN CONFIGURATION INSPECTION")
print("=" * 70)

if not CONFIG_FILE.exists():
    print("Configuration not found:", CONFIG_FILE)
else:
    config_text = CONFIG_FILE.read_text(encoding="utf-8")
    config_lines = config_text.splitlines()

    print("Config path:", CONFIG_FILE)
    print("Total lines:", len(config_lines))

    # Show base config and model-related settings.
    for index, line in enumerate(config_lines):
        if (
            "_base_" in line
            or "model =" in line
            or "backbone" in line
            or "cls_head" in line
            or "num_classes" in line
            or "pretrained" in line
            or "init_cfg" in line
            or "num_segments" in line
            or "input_size" in line
        ):
            start = max(0, index - 1)
            end = min(len(config_lines), index + 3)

            print(f"\nLines {start + 1}-{end}:")
            for j in range(start, end):
                print(f"{j + 1:4}: {config_lines[j]}")

print("\nCELL 8 COMPLETE")

PIP DEPENDENCY DRY RUN
Command: /usr/bin/python3 -m pip install --dry-run --report /content/visafe_pip_dry_run.json mmengine==0.10.7 mmcv-lite==2.1.0 ultralytics

PIP OUTPUT:
Would install addict-2.4.0 mmcv-lite-2.1.0 mmengine-0.10.7 ultralytics-8.4.210 ultralytics-platform-0.1.109 ultralytics-thop-2.2.5 yapf-0.43.0


Dry-run exit code: 0

PROPOSED PACKAGE INSTALLATIONS
mmengine                       0.10.7
mmcv-lite                      2.1.0
ultralytics                    8.4.210
ultralytics-platform           0.1.109
ultralytics-thop               2.2.5
addict                         2.4.0
yapf                           0.43.0

Total packages proposed: 7

GPU package protection check: PASS

TSN CONFIGURATION INSPECTION
Config path: /content/mmaction2_full_v1_2_0/configs/recognition/tsn/tsn_imagenet-pretrained-r50_8xb32-1x1x8-100e_kinetics400-rgb.py
Total lines: 63

Lines 1-3:
   1: _base_ = ['tsn_imagenet-pretrained-r50_8xb32-1x1x3-100e_kinetics400-rgb.py']
   2: 
   3: # dataset se

In [9]:
# ============================================================
# CELL 9: INSTALL VI-SAFE DEPENDENCIES
# ============================================================

import subprocess
import sys
import torch

# Remember the existing GPU stack before installation.
original_torch = torch.__version__

packages = [
    "mmengine==0.10.7",
    "mmcv-lite==2.1.0",
    "ultralytics==8.4.210",
]

print("Installing required packages...")
print("Existing PyTorch:", original_torch)

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--no-cache-dir",
        *packages,
    ],
    check=True,
)

print("\n" + "=" * 65)
print("VERIFY INSTALLED DEPENDENCIES")
print("=" * 65)

# Import and inspect the newly installed packages.
import mmengine
import mmcv
import ultralytics
import torchvision

print("MMEngine     :", mmengine.__version__)
print("MMCV         :", mmcv.__version__)
print("Ultralytics  :", ultralytics.__version__)
print("PyTorch      :", torch.__version__)
print("Torchvision  :", torchvision.__version__)
print("CUDA usable  :", torch.cuda.is_available())

# Verify that PyTorch was not replaced.
assert torch.__version__ == original_torch, (
    f"PyTorch changed unexpectedly: {original_torch} -> {torch.__version__}"
)

assert torch.cuda.is_available(), "CUDA is no longer available."

if torch.cuda.is_available():
    print("GPU          :", torch.cuda.get_device_name(0))

print("\nGPU stack protection: PASSED")
print("DEPENDENCY INSTALLATION COMPLETE")

Installing required packages...
Existing PyTorch: 2.14.1+cu130

VERIFY INSTALLED DEPENDENCIES
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
MMEngine     : 0.10.7
MMCV         : 2.1.0
Ultralytics  : 8.4.210
PyTorch      : 2.14.1+cu130
Torchvision  : 0.29.1+cu130
CUDA usable  : True
GPU          : Tesla T4

GPU stack protection: PASSED
DEPENDENCY INSTALLATION COMPLETE


In [10]:
# ============================================================
# CELL 10: REGISTER LOCAL MMACTION2
# ============================================================

import sys
from pathlib import Path

# ------------------------------------------------------------
# 1. Add the complete MMAction2 repository to Python's path.
# ------------------------------------------------------------
mmaction_path = str(LOCAL_MMACTION)

if mmaction_path not in sys.path:
    sys.path.insert(0, mmaction_path)

# ------------------------------------------------------------
# 2. Import and verify the correct MMAction2 source.
# ------------------------------------------------------------
import mmaction
from mmaction.utils import register_all_modules
from mmaction.registry import MODELS
from mmengine import Config
from mmengine.registry import DefaultScope

print("=" * 65)
print("MMACTION2 REGISTRATION")
print("=" * 65)

print("MMAction2 version :", mmaction.__version__)
print("Imported from     :", mmaction.__file__)
print("Expected source   :", LOCAL_MMACTION)

assert Path(mmaction.__file__).resolve().is_relative_to(
    LOCAL_MMACTION.resolve()
), "Python imported MMAction2 from an unexpected location."

# ------------------------------------------------------------
# 3. Register MMAction2 datasets, models and components.
# ------------------------------------------------------------
register_all_modules(init_default_scope=True)

print("\nModule registration: SUCCESS")

# ------------------------------------------------------------
# 4. Load the project's 8-segment TSN configuration.
# ------------------------------------------------------------
cfg = Config.fromfile(str(CONFIG_FILE))

print("\n" + "=" * 65)
print("TSN CONFIGURATION")
print("=" * 65)

print("Config file :", CONFIG_FILE)
print("Model type  :", cfg.model.type)
print("Backbone    :", cfg.model.backbone.type)
print("Head        :", cfg.model.cls_head.type)

# Print key model settings.
print("\nModel configuration:")
print(cfg.model)

# ------------------------------------------------------------
# 5. Verify registry availability.
# ------------------------------------------------------------
print("\n" + "=" * 65)
print("MODEL REGISTRY CHECK")
print("=" * 65)

for model_type in ["Recognizer2D", "ResNet", "TSNHead"]:
    registered = MODELS.get(model_type)
    print(f"{model_type:20}:",
          "REGISTERED" if registered is not None else "NOT FOUND")

print("\nCELL 10 COMPLETE")

MMACTION2 REGISTRATION
MMAction2 version : 1.2.0
Imported from     : /content/mmaction2_full_v1_2_0/mmaction/__init__.py
Expected source   : /content/mmaction2_full_v1_2_0


/content/mmaction2_full_v1_2_0/mmaction/evaluation/metrics/multimodal_metric.py:27: SyntaxWarning: invalid escape sequence '\d'
  commaStrip = re.compile('(\d)(,)(\d)')  # noqa: W605
/content/mmaction2_full_v1_2_0/mmaction/evaluation/metrics/multimodal_metric.py:28: SyntaxWarning: invalid escape sequence '\d'
  periodStrip = re.compile('(?!<=\d)(\.)(?!\d)')  # noqa: W605
/usr/local/lib/python3.13/dist-packages/mmcv/cnn/bricks/transformer.py:33: UserWarning: Fail to import ``MultiScaleDeformableAttention`` from ``mmcv.ops.multi_scale_deform_attn``, You should install ``mmcv`` rather than ``mmcv-lite`` if you need this module. 
  warnings.warn('Fail to import ``MultiScaleDeformableAttention`` from '


ImportError: cannot import name 'apply_chunking_to_forward' from 'transformers.modeling_utils' (/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py)

In [11]:
# ============================================================
# CELL 11: CHECK TRANSFORMERS 4.44.2 COMPATIBILITY
# ============================================================

import sys
import json
import subprocess
import importlib.metadata as metadata
from pathlib import Path

print("=" * 70)
print("CURRENT TRANSFORMERS ENVIRONMENT")
print("=" * 70)

for package in ["transformers", "huggingface-hub", "tokenizers"]:
    try:
        print(f"{package:20}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package:20}: NOT INSTALLED")

# ------------------------------------------------------------
# Dry-run the compatible Transformers version.
# ------------------------------------------------------------
REPORT_PATH = Path("/content/transformers_compatibility_report.json")

command = [
    sys.executable,
    "-m", "pip", "install",
    "--dry-run",
    "--report", str(REPORT_PATH),
    "transformers==4.44.2",
]

result = subprocess.run(
    command,
    capture_output=True,
    text=True,
)

print("\n" + "=" * 70)
print("TRANSFORMERS 4.44.2 DRY RUN")
print("=" * 70)

print(result.stdout[-8000:] if result.stdout else "(No output)")

if result.stderr:
    print("\nDiagnostics:")
    print(result.stderr[-4000:])

if result.returncode != 0:
    raise RuntimeError("Transformers dependency resolution failed.")

report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
planned = report.get("install", [])

print("\n" + "=" * 70)
print("PROPOSED PACKAGE CHANGES")
print("=" * 70)

for item in planned:
    meta = item.get("metadata", {})
    print(f"{meta.get('name', 'unknown'):30} {meta.get('version', 'unknown')}")

# Protect the existing runtime and successfully installed packages.
protected = {
    "torch",
    "torchvision",
    "torchaudio",
    "numpy",
    "mmengine",
    "mmcv-lite",
    "ultralytics",
}

affected = [
    item.get("metadata", {}).get("name", "").lower()
    for item in planned
    if item.get("metadata", {}).get("name", "").lower() in protected
]

print("\nProtected packages in installation plan:", affected)

if affected:
    print("WARNING: Review the proposed changes before installing.")
else:
    print("Compatibility check: PASS")
    print("No protected packages are proposed for installation.")

print("\nNo packages have been installed or modified.")
print("CELL 11 COMPLETE")

CURRENT TRANSFORMERS ENVIRONMENT
transformers        : 5.22.1
huggingface-hub     : 1.33.0
tokenizers          : 0.23.3

TRANSFORMERS 4.44.2 DRY RUN
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Would install huggingface_hub-0.36.2 tokenizers-0.19.1 transformers-4.44.2


PROPOSED PACKAGE CHANGES
transformers                   4.44.2
huggingface_hub                0.36.2
tokenizers                     0.19.1

Protected packages in installation plan: []
Compatibility check: PASS
No protected packages are proposed for installation.

No packages have been installed or modified.
CELL 11 COMPLETE


In [12]:
# ============================================================
# CELL 12: INSPECT SELECTIVE MMACTION2 REGISTRATION
# ============================================================

from pathlib import Path

MODEL_DIR = LOCAL_MMACTION / "mmaction" / "models"

files_to_inspect = [
    "mmaction/models/__init__.py",
    "mmaction/models/backbones/__init__.py",
    "mmaction/models/heads/__init__.py",
    "mmaction/models/recognizers/__init__.py",
    "mmaction/models/common/__init__.py",
    "mmaction/models/losses/__init__.py",
    "mmaction/models/data_preprocessors/__init__.py",
    "mmaction/models/necks/__init__.py",
    "mmaction/models/multimodal/__init__.py",
    "mmaction/utils/setup_env.py",
]

# ------------------------------------------------------------
# 1. Inspect package initializers and registration logic.
# ------------------------------------------------------------
for relative_path in files_to_inspect:
    path = LOCAL_MMACTION / relative_path

    print("\n" + "=" * 75)
    print(relative_path)
    print("=" * 75)

    if path.exists():
        lines = path.read_text(encoding="utf-8").splitlines()

        # Limit output while retaining relevant import statements.
        for number, line in enumerate(lines[:180], start=1):
            print(f"{number:4}: {line}")

        if len(lines) > 180:
            print(f"... {len(lines) - 180} additional lines omitted")
    else:
        print("FILE NOT FOUND")

# ------------------------------------------------------------
# 2. Inspect the TSN configuration's complete model definition.
# ------------------------------------------------------------
print("\n" + "=" * 75)
print("TSN MODEL CONFIGURATION")
print("=" * 75)

cfg = Config.fromfile(str(CONFIG_FILE))

print(cfg.pretty_text[:12000])

# ------------------------------------------------------------
# 3. Identify registered model components required by TSN.
# ------------------------------------------------------------
print("\n" + "=" * 75)
print("MODEL COMPONENTS REQUIRED BY CONFIG")
print("=" * 75)

def inspect_config_model(model_cfg, prefix="model"):
    if not isinstance(model_cfg, dict):
        return

    model_type = model_cfg.get("type")
    if model_type:
        print(f"{prefix}: {model_type}")

    for key, value in model_cfg.items():
        if isinstance(value, dict):
            inspect_config_model(value, f"{prefix}.{key}")
        elif isinstance(value, (list, tuple)):
            for index, item in enumerate(value):
                if isinstance(item, dict):
                    inspect_config_model(item, f"{prefix}.{key}[{index}]")

inspect_config_model(cfg.model)

print("\nCELL 12 COMPLETE")


mmaction/models/__init__.py
   1: # Copyright (c) OpenMMLab. All rights reserved.
   2: from .backbones import *  # noqa: F401,F403
   3: from .common import *  # noqa: F401,F403
   4: from .data_preprocessors import *  # noqa: F401,F403
   5: from .heads import *  # noqa: F401,F403
   6: from .localizers import *  # noqa: F401,F403
   7: from .losses import *  # noqa: F401,F403
   8: from .multimodal import *  # noqa: F401,F403
   9: from .necks import *  # noqa: F401,F403
  10: from .recognizers import *  # noqa: F401,F403
  11: from .roi_heads import *  # noqa: F401,F403
  12: from .similarity import *  # noqa: F401,F403
  13: from .task_modules import *  # noqa: F401,F403
  14: from .utils import *  # noqa: F401,F403

mmaction/models/backbones/__init__.py
   1: # Copyright (c) OpenMMLab. All rights reserved.
   2: from .aagcn import AAGCN
   3: from .c2d import C2D
   4: from .c3d import C3D
   5: from .mobilenet_v2 import MobileNetV2
   6: from .mobilenet_v2_tsm import MobileNetV

In [13]:
# ============================================================
# CELL 13: INSPECT OPTIONAL MULTIMODAL DEPENDENCY DETECTION
# ============================================================

import sys
from pathlib import Path

DEPENDENCY_FILE = (
    LOCAL_MMACTION / "mmaction" / "utils" / "dependency.py"
)

MULTIMODAL_INIT = (
    LOCAL_MMACTION / "mmaction" / "models" / "multimodal" / "__init__.py"
)

print("=" * 70)
print("MULTIMODAL DEPENDENCY DETECTION")
print("=" * 70)

if DEPENDENCY_FILE.exists():
    dependency_lines = DEPENDENCY_FILE.read_text(
        encoding="utf-8"
    ).splitlines()

    keywords = [
        "WITH_MULTIMODAL",
        "transformers",
        "timm",
        "is_installed",
        "register_multimodal_placeholder",
    ]

    for number, line in enumerate(dependency_lines, start=1):
        if any(keyword.lower() in line.lower() for keyword in keywords):
            print(f"{number:4}: {line}")

else:
    print("dependency.py not found.")

print("\n" + "=" * 70)
print("MULTIMODAL INITIALIZER")
print("=" * 70)

if MULTIMODAL_INIT.exists():
    print(MULTIMODAL_INIT.read_text(encoding="utf-8"))

# ------------------------------------------------------------
# Check whether dependency detection has already been loaded.
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("CURRENT PYTHON MODULE STATE")
print("=" * 70)

for module_name in [
    "mmaction",
    "mmaction.registry",
    "mmaction.utils.dependency",
    "mmaction.models",
    "mmaction.models.multimodal",
]:
    module = sys.modules.get(module_name)

    if module is None:
        print(f"{module_name:35}: NOT LOADED")
    else:
        flag = getattr(module, "WITH_MULTIMODAL", "N/A")
        print(f"{module_name:35}: LOADED | WITH_MULTIMODAL={flag}")

print("\nCELL 13 COMPLETE")

MULTIMODAL DEPENDENCY DETECTION
  42:         dep (str): The dependency package name, like ``transformers``
  43:             or ``transformers>=4.28.0``.
  68: WITH_MULTIMODAL = all(
  69:     satisfy_requirement(item) for item in ['transformers>=4.28.0'])
  72: def register_multimodal_placeholder(names, registry):

MULTIMODAL INITIALIZER
# Copyright (c) OpenMMLab. All rights reserved.
from mmaction.utils.dependency import WITH_MULTIMODAL

if WITH_MULTIMODAL:
    from .vindlu import *  # noqa: F401,F403

else:
    from mmaction.registry import MODELS
    from mmaction.utils.dependency import register_multimodal_placeholder

    register_multimodal_placeholder(
        ['VindLUVQA', 'VindLURetrievalMC', 'VindLURetrieval'], MODELS)


CURRENT PYTHON MODULE STATE
mmaction                           : LOADED | WITH_MULTIMODAL=N/A
mmaction.registry                  : LOADED | WITH_MULTIMODAL=N/A
mmaction.utils.dependency          : LOADED | WITH_MULTIMODAL=True
mmaction.models               

In [14]:
# ============================================================
# CELL 14: SELECTIVE MMACTION2 REGISTRATION
# ============================================================

import sys
import importlib

# ------------------------------------------------------------
# 1. Set the optional multimodal flag before importing models.
# ------------------------------------------------------------
dependency_module = importlib.import_module(
    "mmaction.utils.dependency"
)

print("Original WITH_MULTIMODAL:", dependency_module.WITH_MULTIMODAL)

# Disable only optional VindLU registration.
dependency_module.WITH_MULTIMODAL = False

print("Updated WITH_MULTIMODAL :", dependency_module.WITH_MULTIMODAL)

# ------------------------------------------------------------
# 2. Register MMAction2 modules.
# ------------------------------------------------------------
from mmaction.utils import register_all_modules
from mmaction.registry import MODELS
from mmengine import Config
from mmengine.registry import DefaultScope

register_all_modules(init_default_scope=True)

print("\nMMAction2 registration completed.")

# ------------------------------------------------------------
# 3. Verify the required TSN components.
# ------------------------------------------------------------
required_models = [
    "Recognizer2D",
    "ResNet",
    "TSNHead",
    "AvgConsensus",
    "ActionDataPreprocessor",
]

print("\n" + "=" * 65)
print("TSN COMPONENT REGISTRY")
print("=" * 65)

for name in required_models:
    component = MODELS.get(name)
    print(f"{name:25}:",
          "REGISTERED" if component is not None else "MISSING")

    if component is None:
        raise RuntimeError(f"Required MMAction2 component missing: {name}")

# ------------------------------------------------------------
# 4. Verify default scope.
# ------------------------------------------------------------
scope = DefaultScope.get_current_instance()

print("\nDefault scope:", scope.scope_name if scope else "None")

if scope is None or scope.scope_name != "mmaction":
    raise RuntimeError("MMAction2 default scope was not initialized.")

# ------------------------------------------------------------
# 5. Confirm optional multimodal registration is disabled.
# ------------------------------------------------------------
print("\nOptional multimodal enabled:",
      dependency_module.WITH_MULTIMODAL)

print("\nCELL 14 COMPLETE")

Original WITH_MULTIMODAL: True
Updated WITH_MULTIMODAL : False


KeyError: 'VindLUVQA is already registered in model at mmaction.models.multimodal.vindlu.vindlu_vqa'

In [15]:
# ============================================================
# CELL 15: VALIDATE THE PREVIOUS WORKING DEPENDENCY COMBINATION
# ============================================================

import sys
import json
import subprocess
from pathlib import Path
import torch

REPORT = Path("/content/visafe_previous_dependencies.json")

protected_versions = {
    "torch": torch.__version__,
}

packages = [
    "transformers==4.47.1",
    "tokenizers>=0.21,<0.22",
]

result = subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--dry-run",
        "--report", str(REPORT),
        *packages,
    ],
    capture_output=True,
    text=True,
)

print(result.stdout)

if result.stderr:
    print("\nDiagnostics:")
    print(result.stderr[-3000:])

if result.returncode != 0:
    raise RuntimeError("Dependency dry run failed.")

report = json.loads(REPORT.read_text(encoding="utf-8"))
planned = report.get("install", [])

print("\nPROPOSED INSTALLATIONS")
for item in planned:
    meta = item["metadata"]
    print(f"{meta['name']:25} {meta['version']}")

protected = {
    "torch", "torchvision", "torchaudio",
    "numpy", "mmengine", "mmcv-lite", "ultralytics",
}

affected = [
    item["metadata"]["name"].lower()
    for item in planned
    if item["metadata"]["name"].lower() in protected
]

print("\nProtected packages affected:", affected)
print("Existing PyTorch:", protected_versions["torch"])

if affected:
    raise RuntimeError(
        f"Unexpected changes to protected packages: {affected}"
    )

print("\nDRY RUN PASSED")
print("No packages have been installed.")

  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
Would install huggingface_hub-0.36.2 tokenizers-0.21.4 transformers-4.47.1


PROPOSED INSTALLATIONS
transformers              4.47.1
tokenizers                0.21.4
huggingface_hub           0.36.2

Protected packages affected: []
Existing PyTorch: 2.14.1+cu130

DRY RUN PASSED
No packages have been installed.


In [16]:
# ============================================================
# CELL 16: GITHUB CODE-ONLY SYNC
# ============================================================

from pathlib import Path
from datetime import datetime
from google.colab import userdata
import os
import shutil
import subprocess

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

WORKSPACE = Path("/content/Vi-SAFE-3B42-Implementation")

GITHUB_REPO = "merajsiddieque/Vi-SAFE-3B42-Implementation"
GITHUB_REMOTE = f"https://github.com/{GITHUB_REPO}.git"

SOURCE_DIRECTORIES = {
    "app": WORKSPACE / "app",
    "app_simple": WORKSPACE / "app_simple",
}

# A separate clone keeps Git operations isolated from the training workspace.
SYNC_DIR = Path(
    "/content/visafe_github_sync_"
    + datetime.now().strftime("%Y%m%d_%H%M%S")
)

# ------------------------------------------------------------
# 2. Read credentials from Colab Secrets
# ------------------------------------------------------------

GITHUB_USERNAME = userdata.get("GITHUB_USERNAME")
GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")

if not GITHUB_USERNAME or not GITHUB_TOKEN:
    raise RuntimeError(
        "Missing GITHUB_USERNAME or GITHUB_TOKEN in Colab Secrets."
    )

print("GitHub username secret: FOUND")
print("GitHub token secret    : FOUND")
print("Target repository      :", GITHUB_REPO)

# ------------------------------------------------------------
# 3. Validate local source directories
# ------------------------------------------------------------

for name, source_dir in SOURCE_DIRECTORIES.items():
    if not source_dir.is_dir():
        raise FileNotFoundError(
            f"Source directory missing: {source_dir}"
        )

    python_files = list(source_dir.rglob("*.py"))

    if not python_files:
        raise RuntimeError(f"No Python files found in {source_dir}")

    print(f"{name}: {len(python_files)} Python files found")

# ------------------------------------------------------------
# 4. Create temporary Git authentication helper
# ------------------------------------------------------------

ASKPASS_FILE = Path("/content/visafe_git_askpass.sh")

ASKPASS_FILE.write_text(
    """#!/bin/sh
case "$1" in
    *Username*) printf '%s\\n' "$GITHUB_USERNAME" ;;
    *Password*) printf '%s\\n' "$GITHUB_TOKEN" ;;
    *) exit 1 ;;
esac
""",
    encoding="utf-8",
)

ASKPASS_FILE.chmod(0o700)

git_env = os.environ.copy()
git_env.update({
    "GIT_ASKPASS": str(ASKPASS_FILE),
    "GIT_TERMINAL_PROMPT": "0",
    "GITHUB_USERNAME": GITHUB_USERNAME,
    "GITHUB_TOKEN": GITHUB_TOKEN,
})

def run_git(args, cwd=None):
    """Run Git without printing authentication credentials."""
    result = subprocess.run(
        ["git", *args],
        cwd=cwd,
        env=git_env,
        capture_output=True,
        text=True,
    )

    if result.stdout.strip():
        print(result.stdout.strip())

    if result.returncode != 0:
        # Git output contains no token because the remote URL is clean.
        raise RuntimeError(
            f"Git command failed: git {' '.join(args)}\n"
            f"{result.stderr[-5000:]}"
        )

    return result.stdout.strip()

# ------------------------------------------------------------
# 5. Clone the target repository
# ------------------------------------------------------------

if SYNC_DIR.exists():
    raise FileExistsError(f"Sync directory already exists: {SYNC_DIR}")

print("\nCloning GitHub repository...")

clone_result = subprocess.run(
    [
        "git",
        "-c", "credential.helper=",
        "clone",
        GITHUB_REMOTE,
        str(SYNC_DIR),
    ],
    env=git_env,
    capture_output=True,
    text=True,
)

if clone_result.stdout.strip():
    print(clone_result.stdout.strip())

if clone_result.returncode != 0:
    raise RuntimeError(
        "GitHub clone failed. Check token access and repository permissions.\n"
        + clone_result.stderr[-5000:]
    )

print("Repository cloned successfully.")

# ------------------------------------------------------------
# 6. Identify the branch
# ------------------------------------------------------------

branch = run_git(
    ["branch", "--show-current"],
    cwd=SYNC_DIR,
).strip()

if not branch:
    branch = "main"
    run_git(["checkout", "-b", branch], cwd=SYNC_DIR)

print("Working branch:", branch)

# ------------------------------------------------------------
# 7. Copy ONLY Python source files
# ------------------------------------------------------------

for folder_name, source_dir in SOURCE_DIRECTORIES.items():
    destination = SYNC_DIR / folder_name

    # Replace only the two application source directories.
    if destination.exists():
        shutil.rmtree(destination)

    destination.mkdir(parents=True, exist_ok=True)

    for source_file in source_dir.rglob("*.py"):
        if "__pycache__" in source_file.parts:
            continue

        relative_path = source_file.relative_to(source_dir)
        target_file = destination / relative_path

        target_file.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source_file, target_file)

# ------------------------------------------------------------
# 8. Create strict repository exclusions
# ------------------------------------------------------------

gitignore_content = """
# Python cache
__pycache__/
*.py[cod]
*.pyo
.pytest_cache/
.mypy_cache/
.ruff_cache/

# Virtual environments
.venv/
venv/
env/

# Datasets and videos
RWF-2000/
data/
datasets/
*.mp4
*.avi
*.mov
*.mkv

# Model weights and checkpoints
*.pt
*.pth
*.ckpt
*.onnx
*.engine
*.safetensors

# Training outputs
outputs/
checkpoints/
logs/
metrics/
backups/
wandb/
runs/
*.log

# Notebooks and generated reports
*.ipynb
*.csv
*.parquet

# Secrets and credentials
.env
.env.*
.secrets/
secrets/
credentials/
credentials.json
.git-credentials
*.pem
*.key

# OS/editor files
.DS_Store
Thumbs.db
.vscode/
.idea/
"""

(SYNC_DIR / ".gitignore").write_text(
    gitignore_content.strip() + "\n",
    encoding="utf-8",
)

# ------------------------------------------------------------
# 9. Stage only approved files
# ------------------------------------------------------------

run_git(
    ["add", "--", ".gitignore", "app", "app_simple"],
    cwd=SYNC_DIR,
)

staged_files = run_git(
    ["diff", "--cached", "--name-only"],
    cwd=SYNC_DIR,
).splitlines()

def allowed_file(path):
    if path == ".gitignore":
        return True

    parts = Path(path).parts

    return (
        len(parts) >= 2
        and parts[0] in {"app", "app_simple"}
        and Path(path).suffix == ".py"
        and "__pycache__" not in parts
    )

unexpected_files = [
    path for path in staged_files
    if not allowed_file(path)
]

if unexpected_files:
    raise RuntimeError(
        "Unexpected files were staged; refusing to publish:\n"
        + "\n".join(unexpected_files)
    )

if not staged_files:
    print("No code changes detected.")

else:
    print("\nApproved files staged:")
    for file in staged_files:
        print(" -", file)

    # Check whitespace and patch formatting.
    run_git(["diff", "--cached", "--check"], cwd=SYNC_DIR)

    # --------------------------------------------------------
    # 10. Commit source code
    # --------------------------------------------------------
    run_git(
        ["config", "user.name", GITHUB_USERNAME],
        cwd=SYNC_DIR,
    )
    run_git(
        [
            "config",
            "user.email",
            f"{GITHUB_USERNAME}@users.noreply.github.com",
        ],
        cwd=SYNC_DIR,
    )

    run_git(
        [
            "commit",
            "-m",
            "Add Vi-SAFE implementation source code",
        ],
        cwd=SYNC_DIR,
    )

# ------------------------------------------------------------
# 11. Push to GitHub
# ------------------------------------------------------------

print("\nPushing code to GitHub...")

run_git(
    ["push", "-u", "origin", branch],
    cwd=SYNC_DIR,
)

# Remove the temporary helper file.
ASKPASS_FILE.unlink(missing_ok=True)

print("\n" + "=" * 65)
print("GITHUB CODE SYNC COMPLETED")
print("=" * 65)

print("Repository :", GITHUB_REPO)
print("Branch     :", branch)
print("Files      :", len(staged_files))
print("Published  : app/ and app_simple/ Python source only")
print("Excluded   : datasets, notebooks, weights, outputs and secrets")
print("Git workspace:", SYNC_DIR)
print("\nGitHub:", GITHUB_REMOTE)

GitHub username secret: FOUND
GitHub token secret    : FOUND
Target repository      : merajsiddieque/Vi-SAFE-3B42-Implementation
app: 5 Python files found
app_simple: 5 Python files found

Cloning GitHub repository...
Repository cloned successfully.
main
Working branch: main
.gitignore
app/__init__.py
app/dataset_rwf2000.py
app/encoder_yolo.py
app/main.py
app/yolo_tsn.py
app_simple/__init__.py
app_simple/dataset_rwf2000.py
app_simple/encoder_yolo.py
app_simple/main.py
app_simple/yolo_tsn.py

Approved files staged:
 - .gitignore
 - app/__init__.py
 - app/dataset_rwf2000.py
 - app/encoder_yolo.py
 - app/main.py
 - app/yolo_tsn.py
 - app_simple/__init__.py
 - app_simple/dataset_rwf2000.py
 - app_simple/encoder_yolo.py
 - app_simple/main.py
 - app_simple/yolo_tsn.py
[main (root-commit) eca7ff1] Add Vi-SAFE implementation source code
 11 files changed, 1741 insertions(+)
 create mode 100644 .gitignore
 create mode 100644 app/__init__.py
 create mode 100644 app/dataset_rwf2000.py
 create mod

In [17]:
# ============================================================
# CELL 1 — COPY PROJECT FROM GOOGLE DRIVE TO COLAB
# ============================================================

from pathlib import Path
import shutil
import subprocess

# Source and destination
DRIVE_ROOT = Path("/content/drive/MyDrive/Vi-SAFE-3B42")
LOCAL_ROOT = Path("/content/Vi-SAFE-3B42")

# Verify Google Drive is mounted and source exists
if not Path("/content/drive/MyDrive").exists():
    raise RuntimeError(
        "Google Drive is not mounted. Mount Drive before running this cell."
    )

if not DRIVE_ROOT.is_dir():
    raise FileNotFoundError(f"Project folder not found: {DRIVE_ROOT}")

# Copy the complete project into Colab's local storage
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Copying project:\n{DRIVE_ROOT}\n→ {LOCAL_ROOT}")

shutil.copytree(
    DRIVE_ROOT,
    LOCAL_ROOT,
    dirs_exist_ok=True,
    copy_function=shutil.copy2
)

# Display copied project structure
print("\nCOPY COMPLETED")
print("=" * 60)

for item in sorted(LOCAL_ROOT.iterdir()):
    if item.is_dir():
        print(f"[DIR]  {item.name}/")
    else:
        print(f"[FILE] {item.name}")

# Display local project size
result = subprocess.run(
    ["du", "-sh", str(LOCAL_ROOT)],
    capture_output=True,
    text=True,
    check=True
)

print("\nLocal project size:", result.stdout.strip())
print("Local project path:", LOCAL_ROOT)

Copying project:
/content/drive/MyDrive/Vi-SAFE-3B42
→ /content/Vi-SAFE-3B42


KeyboardInterrupt: 

In [18]:

# ================================================================
# CELL 1 — COPY Vi-SAFE PROJECT FROM GOOGLE DRIVE TO COLAB
# ================================================================

from pathlib import Path
import shutil
import os

# ------------------------------------------------
# 1. PROJECT PATHS
# ------------------------------------------------

DRIVE_ROOT = Path("/content/drive/MyDrive/Vi-SAFE-3B42")
LOCAL_ROOT = Path("/content/Vi-SAFE-3B42-Implementation")

YOLO_SOURCE = DRIVE_ROOT / "yolov8n.pt"
YOLO_DESTINATION = LOCAL_ROOT / "yolov8n.pt"

# ------------------------------------------------
# 2. VERIFY GOOGLE DRIVE AND PROJECT
# ------------------------------------------------

if not Path("/content/drive/MyDrive").exists():
    raise RuntimeError(
        "Google Drive is not mounted. Mount Drive first."
    )

if not DRIVE_ROOT.is_dir():
    raise FileNotFoundError(
        f"Project folder does not exist: {DRIVE_ROOT}"
    )

for required in ["app", "app_simple"]:
    if not (DRIVE_ROOT / required).is_dir():
        raise FileNotFoundError(
            f"Required project folder is missing: {required}"
        )

if not YOLO_SOURCE.is_file():
    raise FileNotFoundError(
        f"YOLO model file not found: {YOLO_SOURCE}"
    )

# ------------------------------------------------
# 3. EXCLUSION RULES
# ------------------------------------------------

# Directories that must not be copied
EXCLUDED_DIRS = {
    "backups",
    "backup",
    "outputs",
    "output",
    "rwf-2000",
    "dataset",
    "datasets",
    "data",
    "mmaction2",
    "checkpoints",
    "checkpoint",
    "saved_models",
    "model_checkpoints",
    "model_weights",
    "weights",
    "runs",
    "wandb",
    "logs",
    "artifacts",
    "cache",
    "__pycache__",
    ".git",
    ".venv",
    "venv",
    "env",
    "node_modules",
    ".ipynb_checkpoints",
}

# File formats that should not be copied
EXCLUDED_EXTENSIONS = {
    # Model weights and checkpoints
    ".pt",
    ".pth",
    ".ckpt",
    ".safetensors",
    ".onnx",
    ".engine",
    ".weights",
    ".pb",
    ".h5",
    ".keras",
    ".tflite",

    # Dataset media
    ".mp4",
    ".avi",
    ".mov",
    ".mkv",
    ".webm",
    ".mpeg",
    ".mpg",
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".wav",
    ".mp3",
    ".flac",

    # Dataset and generated binary data
    ".npy",
    ".npz",
    ".parquet",
    ".arrow",
    ".feather",
    ".pkl",
    ".pickle",
    ".joblib",

    # Archives
    ".zip",
    ".rar",
    ".7z",
    ".tar",
    ".gz",
}

# Credential and secret filenames
EXCLUDED_FILENAMES = {
    ".env",
    "credentials.json",
    "secrets.json",
    "service-account.json",
    "client_secret.json",
    "id_rsa",
    "id_ed25519",
}

EXCLUDED_SECRET_SUFFIXES = {
    ".pem",
    ".key",
    ".p12",
    ".pfx",
}

# Maximum size for an individual copied file
MAX_FILE_SIZE = 20 * 1024 * 1024  # 20 MB

# Only this one model file is allowed locally
ALLOWED_MODEL_FILES = {
    YOLO_SOURCE.resolve()
}

# ------------------------------------------------
# 4. CREATE A CLEAN DESTINATION
# ------------------------------------------------

# This resets the destination so old datasets, weights,
# or previously copied unwanted files cannot remain there.

if LOCAL_ROOT.exists():
    shutil.rmtree(LOCAL_ROOT)

LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------
# 5. COPY PROJECT RECURSIVELY WITH FILTERING
# ------------------------------------------------

copied_files = []
skipped_files = []
copied_bytes = 0

for current_dir, directories, filenames in os.walk(DRIVE_ROOT):

    current_path = Path(current_dir)
    relative_dir = current_path.relative_to(DRIVE_ROOT)

    # Exclude unwanted directories before traversing them
    permitted_directories = []

    for directory in directories:
        if directory.lower() in EXCLUDED_DIRS or directory.startswith("."):
            skipped_files.append(
                (str(relative_dir / directory), "Excluded directory")
            )
        else:
            permitted_directories.append(directory)

    directories[:] = permitted_directories

    destination_dir = LOCAL_ROOT / relative_dir
    destination_dir.mkdir(parents=True, exist_ok=True)

    for filename in filenames:

        source_file = current_path / filename
        relative_file = source_file.relative_to(DRIVE_ROOT)
        resolved_source = source_file.resolve()

        # Preserve .gitignore, skip other hidden files
        if filename.startswith(".") and filename != ".gitignore":
            skipped_files.append((str(relative_file), "Hidden file"))
            continue

        # Never copy credentials
        if filename.lower() in EXCLUDED_FILENAMES:
            skipped_files.append((str(relative_file), "Credential file"))
            continue

        if source_file.suffix.lower() in EXCLUDED_SECRET_SUFFIXES:
            skipped_files.append((str(relative_file), "Private key/certificate"))
            continue

        # Permit only the explicitly allowed YOLO model
        if (
            source_file.suffix.lower() in EXCLUDED_EXTENSIONS
            and resolved_source not in ALLOWED_MODEL_FILES
        ):
            skipped_files.append((str(relative_file), "Excluded file type"))
            continue

        # Do not copy large files, except explicitly allowed files
        file_size = source_file.stat().st_size

        if (
            file_size > MAX_FILE_SIZE
            and resolved_source not in ALLOWED_MODEL_FILES
        ):
            skipped_files.append((str(relative_file), "Larger than 20 MB"))
            continue

        destination_file = destination_dir / filename

        shutil.copy2(source_file, destination_file)

        copied_files.append(str(relative_file))
        copied_bytes += file_size

# ------------------------------------------------
# 6. VERIFY IMPORTANT PROJECT FILES
# ------------------------------------------------

expected_items = [
    "app",
    "app_simple",
    ".gitignore",
    "README.md",
    "requirements.txt",
    "folder_structure.txt",
    "RWF-2000-v1.ipynb",
    "RWF-2000-v2.ipynb",
    "yolov8n.pt",
]

print("\n" + "=" * 65)
print("Vi-SAFE PROJECT COPY REPORT")
print("=" * 65)

for item in expected_items:
    path = LOCAL_ROOT / item
    status = "COPIED" if path.exists() else "NOT PRESENT"
    print(f"{status:15} {item}")

print("\nCopied files:", len(copied_files))
print(f"Copied size: {copied_bytes / (1024 * 1024):.2f} MB")
print("Skipped items:", len(skipped_files))

# ------------------------------------------------
# 7. VERIFY YOLO MODEL
# ------------------------------------------------

if not YOLO_DESTINATION.is_file():
    raise RuntimeError("YOLO model was not copied successfully.")

print(
    f"\nYOLO model copied: "
    f"{YOLO_DESTINATION.stat().st_size / (1024 * 1024):.2f} MB"
)

# ------------------------------------------------
# 8. DISPLAY LOCAL PROJECT STRUCTURE
# ------------------------------------------------

print("\nLOCAL PROJECT STRUCTURE")
print("-" * 65)

for path in sorted(LOCAL_ROOT.rglob("*")):
    relative = path.relative_to(LOCAL_ROOT)

    if path.is_dir():
        print(f"[DIR]  {relative}/")
    else:
        print(f"[FILE] {relative}")

# ------------------------------------------------
# 9. FINAL STATUS
# ------------------------------------------------

print("\n" + "=" * 65)
print("COPY COMPLETED SUCCESSFULLY")
print("=" * 65)

print("Source:      ", DRIVE_ROOT)
print("Destination: ", LOCAL_ROOT)

print("\nExcluded:")
print("- RWF-2000 dataset")
print("- Backups")
print("- Outputs and logs")
print("- Incomplete Drive mmaction2 folder")
print("- Other model weights and checkpoints")
print("- Credentials and private keys")

print("\nKept locally:")
print("- Application source code")
print("- Configuration and documentation")
print("- Both notebooks")
print("- yolov8n.pt")

print("\nNo files in Google Drive were modified.")



Vi-SAFE PROJECT COPY REPORT
COPIED          app
COPIED          app_simple
COPIED          .gitignore
COPIED          README.md
COPIED          requirements.txt
COPIED          folder_structure.txt
COPIED          RWF-2000-v1.ipynb
COPIED          RWF-2000-v2.ipynb
COPIED          yolov8n.pt

Copied files: 15
Copied size: 7.26 MB
Skipped items: 5

YOLO model copied: 6.25 MB

LOCAL PROJECT STRUCTURE
-----------------------------------------------------------------
[FILE] .gitignore
[FILE] README.md
[FILE] RWF-2000-v1.ipynb
[FILE] RWF-2000-v2.ipynb
[DIR]  app/
[FILE] app/dataset_rwf2000.py
[FILE] app/encoder_yolo.py
[FILE] app/main.py
[FILE] app/yolo_tsn.py
[DIR]  app_simple/
[FILE] app_simple/dataset_rwf2000.py
[FILE] app_simple/encoder_yolo.py
[FILE] app_simple/main.py
[FILE] app_simple/yolo_tsn.py
[FILE] folder_structure.txt
[FILE] requirements.txt
[FILE] yolov8n.pt

COPY COMPLETED SUCCESSFULLY
Source:       /content/drive/MyDrive/Vi-SAFE-3B42
Destination:  /content/Vi-SAFE-3B42-Impl

In [ ]:

# ================================================================
# CELL 1 — COPY COMPLETE Vi-SAFE PROJECT FROM DRIVE TO COLAB
# ================================================================

from pathlib import Path
import shutil
import os

# ------------------------------------------------
# 1. PROJECT PATHS
# ------------------------------------------------

DRIVE_ROOT = Path("/content/drive/MyDrive/Vi-SAFE-3B42")
LOCAL_ROOT = Path("/content/Vi-SAFE-3B42-Implementation")

# ------------------------------------------------
# 2. VERIFY GOOGLE DRIVE AND PROJECT
# ------------------------------------------------

if not Path("/content/drive/MyDrive").exists():
    raise RuntimeError(
        "Google Drive is not mounted. Mount Drive first."
    )

if not DRIVE_ROOT.is_dir():
    raise FileNotFoundError(
        f"Project folder does not exist: {DRIVE_ROOT}"
    )

# ------------------------------------------------
# 3. EXCLUSION RULES
# ------------------------------------------------

# Exclude ONLY these three top-level directories.
EXCLUDED_DIRS = {
    "RWF-2000",
    "backups",
    "outputs"
}

# ------------------------------------------------
# 4. RESET LOCAL DESTINATION
# ------------------------------------------------

# Remove previous local copy to avoid stale files.
# This does NOT modify anything in Google Drive.

if LOCAL_ROOT.exists():
    shutil.rmtree(LOCAL_ROOT)

LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------
# 5. COPY EVERYTHING EXCEPT EXCLUDED DIRECTORIES
# ------------------------------------------------

copied_files = []
copied_directories = []
skipped_directories = []
copied_bytes = 0

for item in DRIVE_ROOT.iterdir():

    # Skip only the three specified top-level directories
    if item.is_dir() and item.name in EXCLUDED_DIRS:
        skipped_directories.append(item.name)
        continue

    destination = LOCAL_ROOT / item.name

    if item.is_dir():

        shutil.copytree(item, destination)

        copied_directories.append(item.name)

        for file in destination.rglob("*"):
            if file.is_file():
                relative_path = file.relative_to(LOCAL_ROOT)
                copied_files.append(str(relative_path))
                copied_bytes += file.stat().st_size

    elif item.is_file():

        shutil.copy2(item, destination)

        copied_files.append(item.name)
        copied_bytes += destination.stat().st_size

# ------------------------------------------------
# 6. VERIFY EXCLUDED DIRECTORIES
# ------------------------------------------------

for excluded in EXCLUDED_DIRS:
    if (LOCAL_ROOT / excluded).exists():
        raise RuntimeError(
            f"Excluded directory was unexpectedly copied: {excluded}"
        )

# ------------------------------------------------
# 7. VERIFY IMPORTANT FILES AND DIRECTORIES
# ------------------------------------------------

expected_items = [
    "app",
    "app_simple",
    "mmaction2",
    "README.md",
    "requirements.txt",
    "folder_structure.txt",
    "RWF-2000-v1.ipynb",
    "RWF-2000-v2.ipynb",
    "yolov8n.pt",
]

print("\n" + "=" * 70)
print("Vi-SAFE COMPLETE PROJECT COPY REPORT")
print("=" * 70)

print("\nIMPORTANT ITEMS:")

for item in expected_items:
    path = LOCAL_ROOT / item
    status = "COPIED" if path.exists() else "NOT PRESENT IN SOURCE"
    print(f"[{status}] {item}")

# ------------------------------------------------
# 8. COPY STATISTICS
# ------------------------------------------------

print("\n" + "=" * 70)
print("COPY STATISTICS")
print("=" * 70)

print("Copied top-level directories:", len(copied_directories))
print("Copied files:", len(copied_files))

print(
    f"Total copied size: "
    f"{copied_bytes / (1024 ** 3):.3f} GB"
)

print("\nCopied directories:")

for directory in copied_directories:
    print(f"  [DIR] {directory}/")

print("\nSkipped directories:")

for directory in skipped_directories:
    print(f"  [EXCLUDED] {directory}/")

# ------------------------------------------------
# 9. VERIFY YOLO MODEL
# ------------------------------------------------

YOLO_MODEL = LOCAL_ROOT / "yolov8n.pt"

if not YOLO_MODEL.is_file():
    print("\nWARNING: yolov8n.pt was not found in the source copy.")
else:
    print(
        f"\nYOLOv8 model verified: "
        f"{YOLO_MODEL.stat().st_size / (1024 ** 2):.2f} MB"
    )

# ------------------------------------------------
# 10. DISPLAY COMPLETE LOCAL STRUCTURE
# ------------------------------------------------

print("\n" + "=" * 70)
print("LOCAL PROJECT STRUCTURE")
print("=" * 70)

for path in sorted(LOCAL_ROOT.rglob("*")):
    relative = path.relative_to(LOCAL_ROOT)

    if path.is_dir():
        print(f"[DIR]  {relative}/")
    else:
        print(f"[FILE] {relative}")

# ------------------------------------------------
# 11. FINAL VERIFICATION
# ------------------------------------------------

print("\n" + "=" * 70)
print("COPY COMPLETED")
print("=" * 70)

print("SOURCE:      ", DRIVE_ROOT)
print("DESTINATION: ", LOCAL_ROOT)

print("\nExcluded:")
print("- RWF-2000/")
print("- backups/")
print("- outputs/")

print("\nEverything else has been copied recursively.")
print("Google Drive source files remain unchanged.")
print("=" * 70)
